# 04 — Évaluation et figures
Lire les prédictions persistées, recalculer les métriques et produire les figures du rapport.

In [ ]:
import os
import subprocess
from pathlib import Path

if not Path("../pyproject.toml").exists():
    repo = (Path("/content") if Path("/content").exists() else Path.cwd()) / "iavisuelle"
    if not repo.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/Aymaneerrachidi/iavisuelle.git", str(repo)],
            check=True,
        )
    os.chdir(repo / "notebooks")
print("Working directory:", Path.cwd())

In [ ]:
%pip install -q -e ..[ml,data] matplotlib seaborn

In [ ]:
from pathlib import Path

from vlm_lab.metrics import summarize
from vlm_lab.repository import Repository

repo = Repository(Path("../artifacts/vlm_lab.sqlite3"))
runs = repo.list_experiments()
[(r.id, r.name, r.status) for r in runs[:10]]

In [ ]:
run_id = runs[0].id  # vérifier avant exécution
metrics = summarize(repo.predictions(run_id))
[m.model_dump() for m in metrics]

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

df = pd.DataFrame([m.model_dump() for m in metrics])
Path("../results").mkdir(exist_ok=True)
sns.scatterplot(data=df, x="peak_vram_mb", y="bleu4", hue="model_id", style="precision", s=120)
plt.title("Qualité / mémoire")
plt.tight_layout()
plt.savefig("../results/quality_vs_vram.png", dpi=180)
plt.show()